# TurkishBERTweet - Siber Zorbalık Sınıflandırması
Bu notebook, **TurkishBERTweet** modeli ile siber zorbalık tespiti yapmayı hedefler.

Adil bir kıyaslama için DOCX makalesindeki kriterlere **birebir uygun** olarak kodlanmıştır:
- Epoch: 4
- Learning Rate: 2e-5
- Batch Size: 16
- Max Sequence Length: 256
- Eğitim / Test Ayrımı: %80 / %20

In [ ]:
import pandas as pd
import numpy as np
import torch
import gc
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'*** Kullanılan Cihaz: {device} ***')

In [ ]:
# -----------------------------------------------------
# 1. MODEL VE HİPERPARAMETRELER (Orijinal Makaleye Göre)
# -----------------------------------------------------
EPOCHS = 4
LEARNING_RATE = 2e-5
BATCH_SIZE = 16
MAX_LEN = 256
MODEL_NAME = 'VRLLab/TurkishBERTweet'
RANDOM_STATE = 42

In [ ]:
# -----------------------------------------------------
# 2. VERİ SETİ (DATASET) SINIFI
# -----------------------------------------------------
class CyberbullyingDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, item):
        text = str(self.texts[item])
        label = self.labels[item]
        encoding = self.tokenizer(
            text, add_special_tokens=True, max_length=self.max_len,
            return_token_type_ids=False, padding='max_length',
            truncation=True, return_attention_mask=True, return_tensors='pt'
        )
        return {
            'text': text,
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'targets': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# -----------------------------------------------------
# 3. EĞİTİM VE TEST FONKSİYONLARI
# -----------------------------------------------------
def train_epoch(model, data_loader, optimizer, device, n_examples):
    model = model.train()
    losses = []
    correct_predictions = 0
    for d in data_loader:
        input_ids = d['input_ids'].to(device)
        attention_mask = d['attention_mask'].to(device)
        targets = d['targets'].to(device)
        
        outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
        loss = outputs.loss
        logits = outputs.logits
        _, preds = torch.max(logits, dim=1)
        
        correct_predictions += torch.sum(preds == targets)
        losses.append(loss.item())
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    return correct_predictions.double() / n_examples, np.mean(losses)

def eval_model(model, data_loader, device, n_examples):
    model = model.eval()
    losses = []
    correct_predictions = 0
    real_targets, predictions = [], []
    
    with torch.no_grad():
        for d in data_loader:
            input_ids = d['input_ids'].to(device)
            attention_mask = d['attention_mask'].to(device)
            targets = d['targets'].to(device)
            
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=targets)
            loss = outputs.loss
            logits = outputs.logits
            _, preds = torch.max(logits, dim=1)
            
            correct_predictions += torch.sum(preds == targets)
            losses.append(loss.item())
            real_targets.extend(targets.cpu().numpy())
            predictions.extend(preds.cpu().numpy())
            
    accuracy = accuracy_score(real_targets, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(real_targets, predictions, average='macro', zero_division=0)
    return accuracy, precision, recall, f1, np.mean(losses)

def run_experiment(file_path):
    print(f'\n\n========== [ MESAİ BAŞLADI: {file_path} ] ==========')
    df = pd.read_excel(file_path)
    
    text_col = df.columns[0]
    label_col = 'total' if 'total' in df.columns else df.columns[1]
    df = df.dropna(subset=[text_col, label_col])
    
    df_train, df_test = train_test_split(df, test_size=0.2, random_state=RANDOM_STATE, stratify=df[label_col])
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    
    train_data_loader = DataLoader(CyberbullyingDataset(df_train[text_col].to_numpy(), df_train[label_col].to_numpy(), tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=True)
    test_data_loader = DataLoader(CyberbullyingDataset(df_test[text_col].to_numpy(), df_test[label_col].to_numpy(), tokenizer, MAX_LEN), batch_size=BATCH_SIZE, shuffle=False)
    
    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=LEARNING_RATE)
    
    for epoch in range(EPOCHS):
        print(f'Epoch {epoch + 1}/{EPOCHS} çalışıyor...')
        train_acc, train_loss = train_epoch(model, train_data_loader, optimizer, device, len(df_train))
        print(f'  ✓ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}')
        
    accuracy, precision, recall, f1, test_loss = eval_model(model, test_data_loader, device, len(df_test))
    
    print(f'\n🔥 SONUÇ METRİKLERİ ({file_path}):')
    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    print(f"F1-Score:  {f1:.4f}")
    print("="*60)
    
    del model, optimizer, train_data_loader, test_data_loader
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    gc.collect()
    return accuracy, precision, recall, f1

In [ ]:
# --- ÇALIŞTIRMA BÖLÜMÜ ---
# Kendi yerel projenizdeki dosyaları bulup sırasıyla eğitir. 
pre_covid_metrics = run_experiment('/content/drive/MyDrive/data/covidoncesi.xlsx')
post_covid_metrics = run_experiment('/content/drive/MyDrive/data/covidsonrasi.xlsx')
combined_metrics = run_experiment('/content/drive/MyDrive/data/covidoncesivesonrasi.xlsx')